<a href="https://colab.research.google.com/github/datapuk2-blip/downloader-video2/blob/main/youtube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Structure

To organize the project, create the following directory and file structure:

```
youtube-transcript-app/
├── app.py
├── requirements.txt
├── templates/
│   └── index.html
└── static/
    └── css/
        └── style.css  # Optional: For custom CSS, if any
```

First, create a folder named `youtube-transcript-app` and then the subdirectories `templates` and `static/css` inside it. Then create the files listed above within their respective folders.

## `app.py`

This file contains the Flask backend logic, handling YouTube URL input, transcript generation, and serving the HTML template.

In [5]:
import os
from flask import Flask, request, render_template, send_file, flash, redirect, url_for
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled, NoCharactersFound, TooManyRequests
from urllib.parse import urlparse, parse_qs

app = Flask(__name__)
app.secret_key = 'your_super_secret_key' # Change this to a strong, random key in production

@app.route('/', methods=['GET', 'POST'])
def index():
    transcript_text = None
    youtube_url = ''
    video_id = None

    if request.method == 'POST':
        youtube_url = request.form.get('youtube_url', '').strip()
        if not youtube_url:
            flash('Please enter a YouTube URL.', 'error')
            return render_template('index.html', transcript_text=None, youtube_url=youtube_url)

        video_id = get_youtube_video_id(youtube_url)

        if not video_id:
            flash('Invalid YouTube URL. Please provide a valid YouTube video link.', 'error')
            return render_template('index.html', transcript_text=None, youtube_url=youtube_url)

        try:
            transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-US'])
            transcript_text = "\n".join([entry['text'] for entry in transcript_list])
        except NoTranscriptFound:
            flash('No English transcript found for this video. It might be unavailable or in a different language.', 'error')
        except TranscriptsDisabled:
            flash('Transcripts are disabled for this video.', 'error')
        except NoCharactersFound:
            flash('No characters found in the transcript for this video.', 'error')
        except TooManyRequests:
            flash('Too many requests. Please try again after some time.', 'error')
        except Exception as e:
            flash(f'An unexpected error occurred: {e}', 'error')

    return render_template('index.html', transcript_text=transcript_text, youtube_url=youtube_url, video_id=video_id)

@app.route('/download', methods=['POST'])
def download_transcript():
    transcript_text = request.form.get('transcript_content')
    video_id = request.form.get('video_id')

    if not transcript_text or not video_id:
        flash('No transcript content or video ID provided for download.', 'error')
        return redirect(url_for('index'))

    filename = f'{video_id}_transcript.txt'
    filepath = os.path.join('/tmp', filename) # Using /tmp for temporary storage

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(transcript_text)

    return send_file(filepath, as_attachment=True, download_name=filename)

def get_youtube_video_id(url):
    parsed_url = urlparse(url)
    if parsed_url.hostname in ('www.youtube.com', 'youtube.com'):
        if parsed_url.path == '/watch':
            params = parse_qs(parsed_url.query)
            return params.get('v', [None])[0]
        elif parsed_url.path.startswith('/embed/'):
            return parsed_url.path.split('/embed/')[1].split('?')[0]
        elif parsed_url.path.startswith('/v/'):
            return parsed_url.path.split('/v/')[1].split('?')[0]
    elif parsed_url.hostname in ('youtu.be'):
        return parsed_url.path[1:].split('?')[0]
    return None

if __name__ == '__main__':
    app.run(debug=True)


ModuleNotFoundError: No module named 'youtube_transcript_api'

## `templates/index.html`

This file defines the user interface using Bootstrap 5, providing an input field for the YouTube URL, displaying the transcript, and offering a download button.

In [10]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>YouTube Transcript Generator</title>
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <!-- Optional: Custom CSS -->
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
    <style>
        body {
            background-color: #f8f9fa;
        }
        .container {
            max-width: 900px;
            margin-top: 50px;
            margin-bottom: 50px;
            background-color: #ffffff;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 4px 8px rgba(0,0,0,0.1);
        }
        .form-control:focus {
            box-shadow: none;
            border-color: #0d6efd;
        }
        .transcript-output {
            white-space: pre-wrap;
            word-wrap: break-word;
            background-color: #e9ecef;
            border: 1px solid #ced4da;
            padding: 15px;
            border-radius: 5px;
            min-height: 200px;
            max-height: 400px;
            overflow-y: auto;
        }
        .alert-flash {
            margin-top: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1 class="text-center mb-4">YouTube Transcript Generator</h1>

        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ 'danger' if category == 'error' else 'info' }} alert-dismissible fade show alert-flash" role="alert">
                        {{ message }}
                        <button type="button" class="btn-close" data-bs-dismiss="alert" aria-label="Close"></button>
                    </div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <form method="POST" action="/">
            <div class="mb-3">
                <label for="youtube_url" class="form-label">Enter YouTube Video URL:</label>
                <input type="url" class="form-control" id="youtube_url" name="youtube_url" placeholder="e.g., https://www.youtube.com/watch?v=dQw4w9WgXcQ" value="{{ youtube_url if youtube_url }}" required>
            </div>
            <div class="d-grid gap-2 mb-4">
                <button type="submit" class="btn btn-primary">Get Transcript</button>
            </div>
        </form>

        {% if transcript_text %}
            <div class="mt-4">
                <h3>Transcript:</h3>
                <div class="transcript-output mb-3">
                    {{ transcript_text }}
                </div>
                <form action="/download" method="POST">
                    <input type="hidden" name="transcript_content" value="{{ transcript_text }}">
                    <input type="hidden" name="video_id" value="{{ video_id }}">
                    <button type="submit" class="btn btn-success">Download Transcript (.txt)</button>
                </form>
            </div>
        {% endif %}
    </div>

    <!-- Bootstrap JS Bundle with Popper -->
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>


Writing templates/index.html


FileNotFoundError: [Errno 2] No such file or directory: 'templates/index.html'

### Option 1: Using the `static/css/style.css` file (Recommended)

To add custom styles using an external file, you'll need to create the `style.css` file inside the `static/css` directory. Then, you can add your CSS rules there.

First, create the `static/css/style.css` file:


In [ ]:
!mkdir -p static/css
%%writefile static/css/style.css
/* Add your custom CSS rules here */

body {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}

.container {
    background-color: #ffffff;
    border: 1px solid #e0e0e0;
}

.btn-primary {
    background-color: #007bff;
    border-color: #007bff;
}

.btn-primary:hover {
    background-color: #0056b3;
    border-color: #0056b3;
}

.transcript-output {
    border-radius: 0.25rem;
    background-color: #f1f3f5;
    padding: 1rem;
}


After creating this file, any styles you add here will be applied to your `index.html` due to the `<link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">` tag.

### Option 2: Adding more inline styles

You can also add more CSS directly within the `<style>` tags in your `index.html` file. This is useful for small, page-specific styles or quick adjustments. The current `index.html` already has a `<style>` block. You can expand it or add another one.

## `requirements.txt`

This file lists all the Python dependencies required for the project. You can install them using `pip install -r requirements.txt`.

In [ ]:
%%writefile requirements.txt
Flask==2.3.3
youtube-transcript-api==0.6.2


## How to Run the Application Locally

1.  **Save the files:** Make sure `app.py`, `requirements.txt`, and the `templates/index.html` file are saved in the correct project structure as described above.
2.  **Install dependencies:** Open your terminal or command prompt, navigate to the `youtube-transcript-app` directory, and run:
    ```bash
    pip install -r requirements.txt
    ```
3.  **Run the Flask app:** In the same directory, run:
    ```bash
    python app.py
    ```
4.  **Access the application:** Open your web browser and go to `http://127.0.0.1:5000/`.

### Important Note for Colab:

If you are running this in a Colab environment, you will need to use `ngrok` or a similar service to expose your local Flask server to the internet, as Colab notebooks run in a virtual machine that isn't directly accessible from your browser outside of the Colab environment itself.

Example for ngrok in Colab (run in a separate cell after `app.run(debug=True)` starts):

```python
!pip install pyngrok
from pyngrok import ngrok

# Terminate any previous ngrok tunnels
ngrok.kill()

# Set up a tunnel for port 5000 (Flask's default port)
ngrok_tunnel = ngrok.connect(5000)
print('Public URL:', ngrok_tunnel.public_url)
```


## Deployment Instructions

### Deploying to Render

Render is a cloud platform for hosting all your applications. It's suitable for Flask apps.

1.  **Sign up for Render:** Go to [Render](https://render.com/) and create an account.
2.  **Connect to Git:** Connect your GitHub or GitLab repository where your Flask app code is pushed.
3.  **Create a New Web Service:**
    *   Go to your Render dashboard and click "New > Web Service".
    *   Select your repository.
4.  **Configure your service:**
    *   **Name:** Choose a unique name for your service (e.g., `youtube-transcript-app`).
    *   **Region:** Select a region close to your users.
    *   **Branch:** Specify the branch to deploy (e.g., `main` or `master`).
    *   **Root Directory:** If your code is in a subdirectory, specify it (e.g., `/youtube-transcript-app`).
    *   **Runtime:** `Python 3`
    *   **Build Command:** `pip install -r requirements.txt`
    *   **Start Command:** `gunicorn app:app`
        *   *Note:* Gunicorn is a production-ready WSGI HTTP server for Unix. You'll need to add `gunicorn` to your `requirements.txt` file: `gunicorn==20.1.0`.
    *   **Instance Type:** Choose a suitable instance type (e.g., `Free` for testing, `Starter` for production).
5.  **Environment Variables:** Add `SECRET_KEY` with a strong, random value for `app.secret_key` in your `app.py`.
6.  **Deploy:** Click "Create Web Service". Render will build and deploy your application.

### Deploying to Railway

Railway is another platform that makes it easy to deploy web applications.

1.  **Sign up for Railway:** Go to [Railway](https://railway.app/) and create an account.
2.  **Connect to Git:** Connect your GitHub account and grant access to your repository.
3.  **Create a New Project:**
    *   From your dashboard, click "New Project".
    *   Select "Deploy from GitHub Repo" and choose your repository.
4.  **Configure Deployment:** Railway will often auto-detect your `requirements.txt` and assume a Flask project.
    *   **Build Command:** `pip install -r requirements.txt` (usually auto-detected)
    *   **Start Command:** `gunicorn app:app`
        *   *Note:* Similar to Render, you'll need `gunicorn` in your `requirements.txt`: `gunicorn==20.1.0`.
5.  **Variables:** Add a `SECRET_KEY` environment variable with a strong, random value.
6.  **Domain:** Railway automatically provides a public domain. You can configure a custom domain if needed.
7.  **Deploy:** Railway will automatically build and deploy your application. You can monitor the logs in the Railway dashboard.

**Final `requirements.txt` for deployment:**

```
Flask==2.3.3
youtube-transcript-api==0.6.2
gunicorn==20.1.0
```